# Notebook 05: Data Merge & Instrument Validation
## Building the Final Feature Matrix — 1990–2022

**Author:** Montaha Ghabri | **Project:** Extension of Saadaoui (2026, JCE)

---

### What this notebook does and why

This notebook constructs the analysis-ready datasets for the entire thesis pipeline. It loads Saadaoui's original Stata file, merges external macro and geopolitical controls, validates the instrument strength across progressively richer control sets, and exports tiered datasets for downstream estimation.

### Scope realism: why 15 controls, not 50–100

The original thesis plan (drafted by the supervisor as a skeleton before data exploration) proposed enriching the control set to "50–100 predictors" and using machine learning for automatic selection. After attempting downloads and inspecting coverage, this proved **statistically and practically infeasible**:

- **Statistical limit:** With n = 385 monthly observations, p = 50 would give n/p = 7.7 — marginal even for regularized ML. The DML cross-fitting literature (Chernozhukov et al. 2018) requires n >> p for nuisance estimators to converge. p = 15 (n/p ≈ 25) is safe; p = 50 is not.
- **Data availability:** Free FRED series have coverage gaps (DXY discontinued 2019, EM FX starts 2006). NLP sentiment (FinBERT) covers only 2015+. GDELT event counts are noisy for bilateral dyads. CDS spreads and supply-chain indices require commercial subscriptions.
- **Comparability:** Extending beyond Saadaoui's 2022-02 endpoint would add the COVID recovery, Russia-Ukraine war, and 2022–2024 energy crisis — periods with potentially structural breaks in the PRI-oil relationship. Keeping 1990-2022 maintains direct comparability.

**The honest contribution:** We do not claim "high-dimensional ML." We claim **non-parametric control function estimation** with a moderately enriched control set (15 variables vs. Saadaoui's 4), using DML to relax linearity while keeping identification identical. This is a narrower but defensible contribution.

### Design decisions

**No look-ahead bias.** All external features are lagged t−1 at merge time.

**Cache raw, transform on load.** FRED caches store untransformed level values. Log and diff transforms are applied at read time on every run. This prevents the double-transformation bug: if a cache stores already-differenced values and the pipeline re-applies log+diff, series go through zero producing ±inf.

**Deterministic cache versioning.** Each cached file is tied to its FRED series ID and download timestamp. No heuristic deletion based on value signs.

**Gap variables never enter `macro_frames`.** Variables with coverage gaps (DXY, REER, copper, wheat, EM FX) are fetched separately via `fetch_fred` and stored as standalone series. They are joined only during tier construction, preventing stale transformed data from corrupting the main sample.

**Explicit variable roles.** Every variable is classified before estimation begins. The role dictionary drives tier construction and is loaded by downstream notebooks.

### Output files (`data/final/`)

| File | n | Vars | Window |
|------|---|------|--------|
| `df_baseline.csv` | 385 | 7 | 1990-02 – 2022-02 |
| `df_extended.csv` | 385 | 17 | 1990-02 – 2022-02 |
| `df_dxy.csv` | ~358 | 18 | 1990-02 – 2019-11 |
| `df_reer.csv` | ~336 | 18 | 1994-03 – 2022-02 |
| `df_comm.csv` | ~360 | 19 | 1992-03 – 2022-02 |
| `variable_roles.json` | — | — | — |

**Note:** `df_extended` has 17 variables (not 18) because `l2lwip` was dropped due to near-perfect collinearity with `llwip` (VIF ≈ 670,000). See Cell 9 and Cell 13 for discussion.

## Cell 1: Imports & Path Setup

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
from statsmodels.tools.tools import add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
from pathlib import Path
import requests
from io import BytesIO, StringIO
import json, warnings, os, time, hashlib

warnings.filterwarnings('ignore', category=UserWarning)

ROOT  = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA  = ROOT / 'data'
RAW   = DATA / '02_features' / 'raw'
NLP   = DATA / '03_nlp'
FINAL = DATA / 'final'
FINAL.mkdir(parents=True, exist_ok=True)

print(f'ROOT : {ROOT}')
print(f'DATA : {DATA.exists()} | RAW: {RAW.exists()} | NLP: {NLP.exists()} | FINAL: {FINAL.exists()}')

ROOT : c:\Users\HP\Desktop\replication+contribution
DATA : True | RAW: True | NLP: True | FINAL: True


## Cell 2: Helper Functions

`first_stage_f` replicates Saadaoui's exact first-stage specification: instrument Δ²PRI on log PRI, conditioning on 3 WTI lags, 2 PRI lags, and the passed controls. HC1 heteroskedasticity-robust standard errors throughout.

`fetch_fred_raw` downloads FRED series with **deterministic cache versioning**. The cache file is named `{series_id}_{YYYYMMDD}.csv` where YYYYMMDD is the download date. This prevents the heuristic deletion bug (old version deleted caches based on value signs, which falsely triggered on legitimate negative values like interest rate spreads). Transforms (log, diff) are applied at load time, never at download time.

In [2]:
def D(s): return s.diff()

def stata_month_to_datetime(period):
    if pd.api.types.is_datetime64_any_dtype(period):
        return pd.to_datetime(period)
    base    = pd.Period('1960-01', freq='M')
    numeric = pd.to_numeric(period, errors='coerce')
    return numeric.map(lambda m: (base + int(m)).to_timestamp(how='end') if pd.notna(m) else pd.NaT)

def to_monthly_index(df, date_col):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    df.index = df.index.to_period('M').to_timestamp('M')
    return df

def resample_to_monthly(df, date_col, value_col, agg='mean'):
    df = to_monthly_index(df, date_col)
    try:    return df[[value_col]].resample('ME').agg(agg)
    except: return df[[value_col]].resample('M').agg(agg)

def first_stage_f(df, x='lpri', z='d2pri', controls=None):
    """HC1-robust first-stage F for H₀: π₁=0. Replicates Saadaoui (2026) spec."""
    if controls is None: controls = ['llwip','dllgop','l2lwip','dl2lgop']
    work = df.copy()
    lag_cols = []
    for l in range(1, 4):
        work[f'L{l}_lwti'] = work['lwti'].shift(l); lag_cols.append(f'L{l}_lwti')
    for l in range(1, 3):
        work[f'L{l}_lpri'] = work['lpri'].shift(l); lag_cols.append(f'L{l}_lpri')
    exog = lag_cols + [c for c in controls if c in work.columns]
    fdf  = pd.DataFrame({x: work[x], z: work[z], **{c: work[c] for c in exog}}).dropna()
    X    = add_constant(fdf[[z] + exog], has_constant='add')
    fit  = sm.OLS(fdf[x], X).fit(cov_type='HC1')
    return float(fit.f_test(f'{z} = 0').fvalue)

def first_stage_f_alt(df, z, controls):
    """Same spec but instrument is z. Used for falsification check only."""
    work = df.copy()
    lag_cols = []
    for l in range(1, 4):
        work[f'L{l}_lwti'] = work['lwti'].shift(l); lag_cols.append(f'L{l}_lwti')
    for l in range(1, 3):
        work[f'L{l}_lpri'] = work['lpri'].shift(l); lag_cols.append(f'L{l}_lpri')
    exog = lag_cols + [c for c in controls if c in work.columns]
    fdf  = pd.DataFrame({'lpri': work['lpri'], z: work[z], **{c: work[c] for c in exog}}).dropna()
    X    = add_constant(fdf[[z] + exog], has_constant='add')
    fit  = sm.OLS(fdf['lpri'], X).fit(cov_type='HC1')
    return float(fit.f_test(f'{z} = 0').fvalue)

def load_saadaoui_base():
    cache = DATA / 'cache' / 'Saadaoui_2026_JCE.parquet'
    if cache.exists():
        df = pd.read_parquet(cache)
    else:
        df = pd.read_stata(DATA / 'Saadaoui_2026_JCE.dta')
        cache.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(cache)
    df = df.sort_values('Period').reset_index(drop=True)
    df['Period_dt'] = stata_month_to_datetime(df['Period'])
    df = df.set_index('Period_dt').sort_index()
    df.index = df.index.to_period('M').to_timestamp('M')
    df['dllgop']  = D(df['llgop'])
    df['dl2lgop'] = D(df['l2lgop'])
    return df

def fetch_fred_raw(series_id, name, cache_dir=RAW, log=False, diff=False):
    """
    Download FRED series with deterministic cache versioning.

    Cache file: {series_id}_{YYYYMMDD}.csv where YYYYMMDD is download date.
    Raw (untransformed) values only. Log/diff applied at load time.

    Previous heuristic (delete if any value <= 0) was wrong because:
    - Interest rate spreads can legitimately be negative
    - Log-transformed variables should be positive, but the check conflated
      two different failure modes
    """
    today_str = pd.Timestamp.now().strftime('%Y%m%d')
    cache_path = cache_dir / f'{series_id}_{today_str}.csv'

    # Check for any existing cache for this series (regardless of date)
    existing = sorted(cache_dir.glob(f'{series_id}_*.csv'))

    if existing:
        # Use most recent cache
        cache_path = existing[-1]
        df = pd.read_csv(cache_path)
        dc = 'date' if 'date' in df.columns else 'DATE'
        df[dc] = pd.to_datetime(df[dc], errors='coerce')
        df = df.dropna(subset=[dc]).set_index(dc).sort_index()
        if name not in df.columns:
            df = df.rename(columns={df.columns[0]: name})

        if log:  df[name] = np.log(df[name])
        if diff: df[name] = df[name].diff()
        print(f'  {name:12s} | cache {cache_path.stem[-8:]} | {df.index.min().date()} – {df.index.max().date()} | non-NaN={df[name].notna().sum()}')
        return df[[name]]

    print(f'  {name:12s} | downloading {series_id}...', end=' ')
    for attempt in range(3):
        try:
            r = requests.get(
                f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}',
                timeout=30)
            r.raise_for_status(); break
        except Exception as e:
            if attempt < 2: time.sleep(2)
            else: print(f'FAILED: {e}'); return None

    try:
        df = pd.read_csv(StringIO(r.text))
        df.columns = ['date', name]
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df = df.dropna(subset=['date']).set_index('date').sort_index()
        df = df.loc['1985-01-01':'2022-02-28']
        df = df.resample('ME').last()
        df.index = df.index.to_period('M').to_timestamp('M')
        df[name] = pd.to_numeric(df[name], errors='coerce')
        df.to_csv(cache_path)  # RAW VALUES ONLY
        n_raw = df[name].notna().sum()
        if log:  df[name] = np.log(df[name])
        if diff: df[name] = df[name].diff()
        print(f'OK | {df.index.min().date()} – {df.index.max().date()} | raw n={n_raw} | post-transform non-NaN={df[name].notna().sum()}')
        return df[[name]]
    except Exception as e:
        print(f'PARSE ERROR: {e}'); return None

print('Helpers defined.')

Helpers defined.


## Cell 3: Load Base Dataset

386 raw observations (1990-01 to 2022-02). The .dta contains 51 columns from Saadaoui's original dataset; only 7 are used downstream (lwti, lpri, d2pri, llwip, llgop, l2lwip, l2lgop). `dllgop` and `dl2lgop` are constructed here as first differences, causing 1990-01 to drop — leaving 385 usable observations from 1990-02. This matches Saadaoui's implementation exactly.

**Note on extending to 2026:** We deliberately keep Saadaoui's 2022-02 endpoint. Extending would add the COVID recovery, Russia-Ukraine war, and 2022–2024 energy crisis — periods where the US-China PRI-oil price relationship may have structurally shifted. Maintaining the original window preserves direct comparability and avoids untestable structural break assumptions.

In [3]:
df_base       = load_saadaoui_base()
BASE_CONTROLS = ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']

print(f'Shape     : {df_base.shape}')
print(f'Date range: {df_base.index.min().date()} to {df_base.index.max().date()}')
f_base = first_stage_f(df_base, controls=BASE_CONTROLS)
print(f'Baseline F-stat (core controls only): {f_base:.3f}')

Shape     : (386, 51)
Date range: 1990-01-31 to 2022-02-28
Baseline F-stat (core controls only): 236.185


## Cell 4: Load Full-Coverage Macro Controls from Local Files

Only variables with full 1990–2022 coverage are loaded here. Gap variables (DXY, REER, copper, wheat, EM FX) are fetched separately in Cell 4b.

### Variable selection rationale

**Included (`controls_macro`):**

| Variable | Transform | Justification |
|----------|-----------|---------------|
| `vix` | Level | Global risk appetite; standard in commodity regressions |
| `gs10` | Level | US long rate; monetary conditions affect commodity demand |
| `tb3ms` | Level | Short rate; yield curve shape |
| `baa10y` | Level | Corporate credit spread; broad financial conditions |
| `brent` | Log-diff | Brent–WTI differential; near-collinear with WTI (r>0.95), monitored |
| `gold` | Log-diff | Safe-haven demand; flight-to-quality channel |
| `bdi` | Log-diff | Baltic Dry Index; global trade volume and shipping costs |
| `cny_usd` | Log-diff | RMB/USD; directly relevant to US-China dyad |
| `indpro` | Log-diff | US industrial production; demand-side control |

**Excluded from estimation (loaded for completeness only):**
- `tedrate` — FRED discontinued Jan 2022; would silently trim sample to n=384
- `us_spread` — VIF=∞; perfect collinearity with `baa10y`
- `unemp` — slow-moving; weak channel to oil at monthly frequency
- `cpi` — partially downstream of oil; risks absorbing the effect being estimated

**Not loaded at all (removed from pipeline):**
- `sentiment` — 54% missing (210/386 NaNs); structurally unusable. Removed to reduce clutter.

In [4]:
# ── Variables with known coverage gaps — excluded here, fetched in Cell 4b ──
GAP_VARS = {'dxy', 'reer', 'copper', 'wheat', 'em_fx'}

def load_macro_control(name, config):
    """Load a full-coverage macro control from local CSV. Gap vars must not be passed."""
    assert name not in GAP_VARS, f"{name} is a gap variable — use fetch_fred instead"
    path = RAW / config['file']
    if not path.exists():
        print(f'  SKIP {name}: file not found'); return None
    try:
        df_raw = pd.read_csv(path)
    except Exception as e:
        print(f'  SKIP {name}: read error — {e}'); return None

    date_col = config['date_col']
    if date_col not in df_raw.columns:
        alts  = ['DATE','Date','date','observation_date','period','Period']
        found = [c for c in df_raw.columns if c in alts]
        if found: date_col = found[0]
        else: print(f'  SKIP {name}: no date column'); return None

    try:
        df_m = resample_to_monthly(df_raw, date_col, config['value_col'], config['agg'])
    except Exception as e:
        print(f'  SKIP {name}: resample failed — {e}'); return None

    if config.get('log', False):
        df_m[f'l{name}'] = np.log(df_m[config['value_col']])
        df_m[name] = df_m[f'l{name}'].diff() if config.get('diff', False) else df_m[f'l{name}']
    else:
        df_m[name] = df_m[config['value_col']].diff() if config.get('diff', False) else df_m[config['value_col']]

    return df_m[[name]]


MACRO_CONFIGS = {
    # ── Full-coverage controls (enter df_extended) ────────────────────────────
    'vix':      {'file': 'vix.csv',          'date_col': 'Date',             'value_col': 'vix',          'agg': 'mean'},
    'gs10':     {'file': 'gs10.csv',         'date_col': 'observation_date', 'value_col': 'GS10',         'agg': 'last'},
    'tb3ms':    {'file': 'tb3ms.csv',        'date_col': 'observation_date', 'value_col': 'TB3MS',        'agg': 'last'},
    'baa10y':   {'file': 'baa10y.csv',       'date_col': 'observation_date', 'value_col': 'BAA10Y',       'agg': 'last'},
    'brent':    {'file': 'brent_fred.csv',   'date_col': 'observation_date', 'value_col': 'DCOILBRENTEU', 'agg': 'mean', 'log': True, 'diff': True},
    'gold':     {'file': 'gold_monthly.csv', 'date_col': 'Date',             'value_col': 'Price',        'agg': 'last', 'log': True, 'diff': True},
    'bdi':      {'file': 'bdi_clean.csv',    'date_col': 'date',             'value_col': 'bdi',          'agg': 'mean', 'log': True, 'diff': True},
    'cny_usd':  {'file': 'cny_usd.csv',      'date_col': 'observation_date', 'value_col': 'DEXCHUS',      'agg': 'last', 'log': True, 'diff': True},
    'indpro':   {'file': 'indpro.csv',       'date_col': 'observation_date', 'value_col': 'INDPRO',       'agg': 'last', 'log': True, 'diff': True},
    # ── Diagnostic only — loaded for completeness, excluded from estimation ──
    'tedrate':  {'file': 'tedrate.csv',      'date_col': 'observation_date', 'value_col': 'TEDRATE',      'agg': 'mean'},
    'us_spread':{'file': 'us_spread.csv',    'date_col': 'observation_date', 'value_col': 'us_spread',    'agg': 'last'},
    'unemp':    {'file': 'unemp.csv',        'date_col': 'date',             'value_col': 'unemp',        'agg': 'last'},
    'cpi':      {'file': 'cpi.csv',          'date_col': 'date',             'value_col': 'cpi',          'agg': 'last'},
}

macro_frames = {}
for name, cfg in MACRO_CONFIGS.items():
    try:
        df_m = load_macro_control(name, cfg)
        if df_m is not None:
            macro_frames[name] = df_m
            nans = df_m[name].isna().sum()
            print(f'  {name:12s}: {df_m.index.min().date()} – {df_m.index.max().date()} | NaNs={nans}')
    except Exception as e:
        print(f'  ERROR {name}: {e}')

print(f'Loaded {len(macro_frames)} / {len(MACRO_CONFIGS)} macro controls.')

  vix         : 1990-01-31 – 2022-02-28 | NaNs=0
  gs10        : 1953-04-30 – 2026-03-31 | NaNs=0
  tb3ms       : 1934-01-31 – 2026-03-31 | NaNs=0
  baa10y      : 1986-01-31 – 2026-04-30 | NaNs=0
  brent       : 1987-05-31 – 2026-04-30 | NaNs=1
  gold        : 1833-01-31 – 2026-03-31 | NaNs=1
  bdi         : 1986-12-31 – 2025-05-31 | NaNs=1
  cny_usd     : 1981-01-31 – 2026-04-30 | NaNs=1
  indpro      : 1919-01-31 – 2026-03-31 | NaNs=1
  tedrate     : 1986-01-31 – 2022-01-31 | NaNs=0
  us_spread   : 1986-01-31 – 2026-04-30 | NaNs=0
  unemp       : 1948-01-31 – 2026-04-30 | NaNs=1
  cpi         : 1990-01-31 – 2022-02-28 | NaNs=0
Loaded 13 / 13 macro controls.


## Cell 4b: Fetch Gap Variables from FRED (Deterministic Cache)

Gap variables are fetched with strict raw-value caching. Cache files are named `{series_id}_{YYYYMMDD}.csv` — deterministic and versioned. No heuristic deletion.

| Variable | FRED Series | Coverage | Notes |
|----------|-------------|----------|-------|
| `dxy` | DTWEXM | 1973–2019-12 | Discontinued |
| `reer` | RBUSBIS | 1994-01+ | BIS real effective exchange rate |
| `copper` | PCOPPUSDM | 1992-01+ | IMF commodity price |
| `wheat` | PWHEAMTUSDM | 1992-01+ | IMF commodity price |
| `em_fx` | DTWEXEMEGS | 2006-01+ | Emerging market FX index |

These are stored as standalone series objects (`gap_series` dict). They never enter `df_merged` or `macro_frames`. Robustness tiers are built by joining from `gap_series` directly onto `df_extended`.

In [5]:
print('GAP VARIABLE FETCH')
print('=' * 60)

GAP_FRED = {
    'dxy':   ('DTWEXM',      True, True),
    'reer':  ('RBUSBIS',     True, True),
    'copper':('PCOPPUSDM',   True, True),
    'wheat': ('PWHEAMTUSDM', True, True),
    'em_fx': ('DTWEXEMEGS',  True, True),
}

gap_series = {}
for name, (sid, log, diff) in GAP_FRED.items():
    s = fetch_fred_raw(sid, name, log=log, diff=diff)
    if s is not None:
        gap_series[name] = s
        print(f'    coverage: {s.dropna().index.min().date()} – {s.dropna().index.max().date()}')

print(f'Gap series available: {list(gap_series.keys())}')

GAP VARIABLE FETCH
  dxy          | cache 20260513 | 1985-01-31 – 2019-12-31 | non-NaN=419
    coverage: 1985-02-28 – 2019-12-31
  reer         | cache 20260513 | 1994-01-31 – 2022-02-28 | non-NaN=337
    coverage: 1994-02-28 – 2022-02-28
  copper       | cache 20260513 | 1992-01-31 – 2022-02-28 | non-NaN=361
    coverage: 1992-02-29 – 2022-02-28
  wheat        | cache 20260513 | 1992-01-31 – 2022-02-28 | non-NaN=361
    coverage: 1992-02-29 – 2022-02-28
  em_fx        | cache 20260513 | 2006-01-31 – 2022-02-28 | non-NaN=193
    coverage: 2006-02-28 – 2022-02-28
Gap series available: ['dxy', 'reer', 'copper', 'wheat', 'em_fx']


## Cell 5: Load UCT (US-China Tension Index)

**Source:** Rogers, Yazdi & Zhu (2024) — policyuncertainty.com

**Role: alternative instrument / heterogeneity variable — NOT a control.**

Including UCT as a control would absorb variation we want to identify with d2pri. It appears in: (1) falsification check (Cell 12), and (2) heterogeneity analysis (Notebook 08). Partial correlation with d2pri: r ≈ −0.02 (p = 0.68). No absorption risk.

In [6]:
def download_uct(cache_path=None):
    if cache_path is None: cache_path = RAW / 'uct.csv'
    if cache_path.exists():
        print('UCT: loading from cache')
        df = pd.read_csv(cache_path)
    else:
        print('UCT: downloading from policyuncertainty.com...')
        r = requests.get('https://www.policyuncertainty.com/media/UCT.csv', timeout=30)
        r.raise_for_status()
        lines = r.text.strip().split('')
        start = next(i for i, l in enumerate(lines)
                     if l.strip().replace('"','') and
                     (l.strip().startswith('19') or l.strip().startswith('20')))
        df = pd.read_csv(BytesIO(''.join(lines[start:]).encode()))
        df.to_csv(cache_path, index=False)

    df  = df.loc[:, df.columns.notna() & (df.columns != '')]
    dc  = df.columns[0]
    vc  = next((c for c in df.columns[1:] if pd.api.types.is_numeric_dtype(df[c])), None)
    if vc is None: raise ValueError(f'No numeric col in UCT. Columns: {df.columns.tolist()}')
    df['date'] = pd.to_datetime(df[dc].astype(str).str.replace('m', '-'),
                                 format='%Y-%m', errors='coerce')
    df = df.dropna(subset=['date']).set_index('date').sort_index()
    df.index = df.index.to_period('M').to_timestamp('M')
    return df[[vc]].rename(columns={vc: 'uct_tension'})

df_uct = download_uct()
print(f'UCT: {df_uct.index.min().date()} to {df_uct.index.max().date()} | n={len(df_uct)}')

UCT: loading from cache
UCT: 1993-02-28 to 2024-02-29 | n=373


## Cell 6: Load GPR (Geopolitical Risk Index)

**Source:** Caldara & Iacoviello (2022)

Three series: global GPR, China-specific, US-specific. `gpr_chn_l1` and `gpr_usa_l1` enter as controls. `gpr_global_l1` is used only for the falsification check.

In [7]:
def download_gpr(cache_path=None):
    if cache_path is None: cache_path = RAW / 'gpr.parquet'
    if cache_path.exists():
        print('GPR: loading from cache'); return pd.read_parquet(cache_path)
    print('GPR: downloading...')
    r = requests.get(
        'https://www.matteoiacoviello.com/gpr_files/data_gpr_export.xls', timeout=30)
    r.raise_for_status()
    df = pd.read_excel(BytesIO(r.content), header=0)
    df.columns = [str(c).strip() for c in df.columns]
    target = {'GPR': 'gpr_global', 'GPRC_CHN': 'gpr_chn', 'GPRC_USA': 'gpr_usa'}
    cols_map = {}
    for orig, new in target.items():
        m = [c for c in df.columns if c.upper().replace(' ', '_') == orig]
        if m: cols_map[m[0]] = new; print(f'  {orig} → {new}')
    dc = df.columns[0]
    df[dc] = pd.to_datetime(df[dc], errors='coerce')
    df = df.dropna(subset=[dc]).set_index(dc).sort_index()
    df.index = df.index.to_period('M').to_timestamp('M')
    df_out = df[list(cols_map.keys())].rename(columns=cols_map)
    df_out = df_out.dropna(how='all').loc['1985':]
    df_out.to_parquet(cache_path)
    print(f'GPR: {df_out.index.min().date()} to {df_out.index.max().date()} | n={len(df_out)}')
    return df_out

# Always rebuild GPR cache (stale cache caused wrong column mapping in previous versions)
for old in [RAW / 'gpr.csv', RAW / 'gpr.parquet']:
    if old.exists(): os.remove(old); print(f'Removed: {old.name}')

df_gpr = download_gpr()
print(f'Columns: {df_gpr.columns.tolist()}')
print(df_gpr.loc['1990-01':'1990-03'])

Removed: gpr.parquet
GPR: downloading...
  GPR → gpr_global
  GPRC_CHN → gpr_chn
  GPRC_USA → gpr_usa
GPR: 1985-01-31 to 2026-04-30 | n=496
Columns: ['gpr_global', 'gpr_chn', 'gpr_usa']
            gpr_global   gpr_chn   gpr_usa
month                                     
1990-01-31   81.544044  0.233520  1.921422
1990-02-28   77.407211  0.169666  1.915435
1990-03-31   67.591942  0.112630  1.604008


## Cell 7: Load NLP Features (Diagnostic Only)

### Why NLP does not enter estimation

The original plan included NLP-derived sentiment as a structural contribution. After data exploration, this proved infeasible:

- **FinBERT** (`finbert_monthly.csv`) — not found. When available, coverage began ~2015 (84/385 months). Structurally unusable: post-2015 US-China relations (trade war, COVID decoupling) are not representative of the full 1990–2022 dynamic.
- **GDELT** (`gdelt_monthly.csv`) — 385 obs, full coverage. But pre-2000 bilateral event density is low, many zeros in early years, and GDELT coding is noisy for bilateral relationships. Presented descriptively only.
- **EA-GPR** (`ea_gpr_monthly.csv`) — 289 obs (1998-02+). Conceptually redundant with `gpr_global_l1` already in the feature matrix.

**Conclusion:** All three NLP features are diagnostic-only. Documented as a limitation and direction for future work with commercial data sources. This is an honest methodological boundary, not a pipeline failure.

In [8]:
def load_nlp_feature(name, file, value_col, date_col='Unnamed: 0', shift=1):
    path = NLP / file
    if not path.exists():
        print(f'  SKIP {name}: not found'); return None
    df = pd.read_csv(path)
    if 'date' in df.columns:
        if date_col in df.columns and date_col != 'date':
            df = df.drop(columns=[date_col])
    elif date_col in df.columns:
        df = df.rename(columns={date_col: 'date'})
    else:
        print(f'  SKIP {name}: no date column'); return None
    if isinstance(df['date'], pd.DataFrame):
        df = df.loc[:, ~df.columns.duplicated()]
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date'])
    df = to_monthly_index(df, 'date')
    if value_col not in df.columns:
        m = [c for c in df.columns if value_col.lower() in c.lower()]
        value_col = m[0] if m else None
    if value_col is None:
        print(f'  SKIP {name}: value col not found'); return None
    return df[[value_col]].rename(columns={value_col: name}).shift(shift)

nlp_features = {}
for name, cfg in {
    'gdelt_events': {'file': 'gdelt_monthly.csv',  'value_col': 'gdelt_n_events'},
    'finbert_net':  {'file': 'finbert_monthly.csv', 'value_col': 'finbert_net'},
    'ea_gpr':       {'file': 'ea_gpr_monthly.csv',  'value_col': 'ea_gpr'},
}.items():
    df_nlp = load_nlp_feature(name, **cfg)
    if df_nlp is not None:
        nlp_features[name] = df_nlp
        nn = df_nlp[name].notna().sum()
        s  = df_nlp.dropna().index.min().date() if nn > 0 else 'N/A'
        e  = df_nlp.dropna().index.max().date() if nn > 0 else 'N/A'
        print(f'  {name:15s}: {s} to {e} | non-NaN={nn}/{len(df_nlp)}')

print(f'Loaded {len(nlp_features)} NLP features (diagnostic only — not used in estimation).')

  gdelt_events   : 1990-02-28 to 2022-02-28 | non-NaN=385/386
  SKIP finbert_net: not found
  ea_gpr         : 1998-02-28 to 2022-02-28 | non-NaN=289/386
Loaded 2 NLP features (diagnostic only — not used in estimation).


## Cell 8: Merge Core Data Sources into `df_merged`

Left-join onto the base Saadaoui matrix. Every external feature is lagged t−1. Gap variables are **not** joined here — they remain in `gap_series` and are joined only during tier construction in Cell 10.

In [9]:
df_merged = df_base.copy()

# Full-coverage macro controls (lagged t-1)
for name, df_m in macro_frames.items():
    df_merged = df_merged.join(df_m.shift(1), how='left', rsuffix='_dup')
    dup = f'{name}_dup'
    if dup in df_merged.columns: df_merged = df_merged.drop(columns=[dup])

# UCT (lagged t-1)
df_merged = df_merged.join(
    df_uct.shift(1).rename(columns={'uct_tension': 'uct_tension_l1'}),
    how='left', rsuffix='_dup')
if 'uct_tension_l1_dup' in df_merged.columns:
    df_merged = df_merged.drop(columns=['uct_tension_l1_dup'])

# GPR (lagged t-1)
df_gpr_lag = df_gpr.shift(1).rename(columns={
    'gpr_global': 'gpr_global_l1',
    'gpr_chn':    'gpr_chn_l1',
    'gpr_usa':    'gpr_usa_l1'})
df_merged = df_merged.join(df_gpr_lag, how='left', rsuffix='_dup')
for c in ['gpr_global_l1_dup', 'gpr_chn_l1_dup', 'gpr_usa_l1_dup']:
    if c in df_merged.columns: df_merged = df_merged.drop(columns=[c])

# NLP features (diagnostic; already lagged in load function)
for name, df_nlp in nlp_features.items():
    df_merged = df_merged.join(df_nlp, how='left', rsuffix='_dup')
    if f'{name}_dup' in df_merged.columns:
        df_merged = df_merged.drop(columns=[f'{name}_dup'])

print(f'Merge complete: {df_merged.shape}')
print(f'Date range   : {df_merged.index.min().date()} to {df_merged.index.max().date()}')
print(f'Note: gap variables (dxy, reer, copper, wheat, em_fx) are NOT in df_merged.')
print(f'      They will be joined per-tier in Cell 10.')

Merge complete: (386, 70)
Date range   : 1990-01-31 to 2022-02-28
Note: gap variables (dxy, reer, copper, wheat, em_fx) are NOT in df_merged.
      They will be joined per-tier in Cell 10.


## Cell 9: Variable Roles

### Critical change from previous version: `l2lwip` dropped from `controls_core`

**Why:** `llwip` (log world industrial production, t−1) and `l2lwip` (log world industrial production, t−2) have VIF ≈ 670,000 — near-perfect collinearity. In OLS, this makes the variance-covariance matrix numerically singular; in DML/XGBoost, it causes arbitrary variable selection where one variable gets zero importance and the other absorbs all. This is not "benign" — it is a structural flaw.

**Fix:** Drop `l2lwip` from the extended control set. The lag structure is already captured by the 3 WTI lags and 2 PRI lags in the first-stage specification (Cell 2). `llwip` as a single contemporaneous control is sufficient. `dl2lgop` (first difference of l2lgop) is retained because it is a difference term, not a level, and provides distinct variation.

**Note:** We keep both dllgop and dl2lgop because first differences of persistent series are not collinear with each other (VIF ≈ 1.1 each), unlike the levels which have VIF ≈ 670,000.

**Result:** `controls_core` = 3 variables (`llwip`, `dllgop`, `dl2lgop`). Total extended controls = 14 (3 core + 9 macro + 2 geopol).

In [10]:
VARIABLE_ROLES = {
    'instrument_core': ['d2pri'],
    'instrument_alt':  ['uct_tension_l1', 'gpr_global_l1'],
    'treatment':       ['lpri'],
    'outcome':         ['lwti'],
    'controls_core':   ['llwip', 'dllgop', 'dl2lgop'],  # l2lwip DROPPED — VIF=670k
    'controls_macro':  ['vix', 'gs10', 'tb3ms', 'baa10y', 'brent', 'gold', 'bdi', 'cny_usd', 'indpro'],
    'controls_geopol': ['gpr_chn_l1', 'gpr_usa_l1'],
    'heterogeneity':   ['vix', 'uct_tension_l1', 'gpr_global_l1'],
    'diagnostic_only': [
        'tedrate', 'us_spread', 'unemp', 'cpi',        # coverage/collinearity/endogeneity
        'dxy', 'reer', 'copper', 'wheat', 'em_fx',    # gap vars — robustness tiers only
        'gdelt_events', 'finbert_net', 'ea_gpr',       # NLP — coverage insufficient
    ]
}

print('VARIABLE ROLES')
print('=' * 58)
for role, vlist in VARIABLE_ROLES.items():
    in_merged = sum(1 for v in vlist if v in df_merged.columns)
    in_gap    = sum(1 for v in vlist if v in gap_series)
    print(f'  {role:20s}: {len(vlist):2d} vars  ({in_merged} in df_merged, {in_gap} in gap_series)')
    for v in vlist:
        if v in df_merged.columns:    loc = '✓ merged'
        elif v in gap_series:         loc = '✓ gap_series'
        elif v in nlp_features:       loc = '✓ nlp_features'
        elif v in macro_frames:       loc = '✓ macro_frames'
        else:                         loc = '✗ MISSING'
        print(f'      {v:24s} {loc}')

VARIABLE ROLES
  instrument_core     :  1 vars  (1 in df_merged, 0 in gap_series)
      d2pri                    ✓ merged
  instrument_alt      :  2 vars  (2 in df_merged, 0 in gap_series)
      uct_tension_l1           ✓ merged
      gpr_global_l1            ✓ merged
  treatment           :  1 vars  (1 in df_merged, 0 in gap_series)
      lpri                     ✓ merged
  outcome             :  1 vars  (1 in df_merged, 0 in gap_series)
      lwti                     ✓ merged
  controls_core       :  3 vars  (3 in df_merged, 0 in gap_series)
      llwip                    ✓ merged
      dllgop                   ✓ merged
      dl2lgop                  ✓ merged
  controls_macro      :  9 vars  (9 in df_merged, 0 in gap_series)
      vix                      ✓ merged
      gs10                     ✓ merged
      tb3ms                    ✓ merged
      baa10y                   ✓ merged
      brent                    ✓ merged
      gold                     ✓ merged
      bdi              

## Cell 10: Build Tiered Datasets

### Design rationale

`df_extended` is the main estimation dataset. It uses only variables with full 1990–2022 coverage, so `.dropna()` returns exactly 385 observations.

Tiers 2–4 handle coverage-limited gap variables. **Critical fix from previous version:** `build_gap_tier` now applies `shift(1)` **before** `dropna()`, not after. The old order (`dropna()` then `shift(1)`) collapsed interior NaNs incorrectly: if a gap variable had valid values at t and t−2 but NaN at t−1, the old code would drop t−1, shift the t value to t−1 position, and create a false t−1 lag from t−2. The new order preserves the true time index.

**Expected tier sizes:**
- `df_dxy`: ~358 obs (DXY ends Dec 2019 → after t−1 lag, last usable Nov 2019)
- `df_reer`: ~336 obs (REER starts Jan 1994 → after t−1 lag, first usable Feb 1994)
- `df_comm`: ~360 obs (copper/wheat start Jan 1992 → after t−1 lag, first usable Feb 1992)

In [11]:
full_macro = VARIABLE_ROLES['controls_macro']

# ── Tier 0: Baseline ────────────────────────────────────────────────────────
df_baseline = df_merged[
    ['lwti', 'lpri', 'd2pri'] + VARIABLE_ROLES['controls_core']
].dropna()

# ── Tier 1: Extended (MAIN ESTIMATION DATASET) ──────────────────────────────
extended_cols = (
    ['lwti', 'lpri', 'd2pri'] +
    VARIABLE_ROLES['controls_core'] +
    full_macro +
    VARIABLE_ROLES['controls_geopol']
)
df_extended = df_merged[extended_cols].dropna()

# ── Tiers 2–4: gap variables joined from gap_series ───────────────────────────
def build_gap_tier(base_df, gap_names, window_start=None, window_end=None):
    """
    Join gap variables from gap_series onto base_df.
    CRITICAL: shift(1) BEFORE dropna() to preserve true t-1 lag structure.
    Previous version did dropna() then shift(1), which corrupted interior NaNs.
    """
    tier = base_df.copy()
    for gname in gap_names:
        if gname not in gap_series:
            print(f'  WARNING: {gname} not in gap_series — skipped')
            continue
        # Lag FIRST, then drop NaN — preserves time index integrity
        gs_lag = gap_series[gname].shift(1).dropna()
        tier = tier.join(gs_lag, how='inner')
    if window_start: tier = tier.loc[window_start:]
    if window_end:   tier = tier.loc[:window_end]
    return tier.dropna()

df_dxy  = build_gap_tier(df_extended, ['dxy'],            window_end='2019-11-30')
df_reer = build_gap_tier(df_extended, ['reer'],           window_start='1994-02-01')
df_comm = build_gap_tier(df_extended, ['copper', 'wheat'], window_start='1992-02-01')

datasets = {
    'baseline': df_baseline,
    'extended': df_extended,
    'dxy':      df_dxy,
    'reer':     df_reer,
    'comm':     df_comm,
}

print('TIERED DATASETS')
print('=' * 72)
print(f'{"Tier":6s} {"Dataset":12s} {"n":>5s} {"Vars":>5s} {"Start":12s} {"End"}')
print('-' * 72)
for i, (nm, df) in enumerate(datasets.items()):
    print(f'{i:6d} {nm:12s} {df.shape[0]:5d} {df.shape[1]:5d} ' +
          f'{str(df.index.min().date()):12s} {df.index.max().date()}')
print()
print('Expected: baseline=385, extended=385, dxy~358, reer~336, comm~360')

TIERED DATASETS
Tier   Dataset          n  Vars Start        End
------------------------------------------------------------------------
     0 baseline       385     6 1990-02-28   2022-02-28
     1 extended       385    17 1990-02-28   2022-02-28
     2 dxy            358    18 1990-02-28   2019-11-30
     3 reer           336    18 1994-03-31   2022-02-28
     4 comm           360    19 1992-03-31   2022-02-28

Expected: baseline=385, extended=385, dxy~358, reer~336, comm~360


## Cell 11: Instrument Validation — First-Stage F-Statistics

Weak instrument threshold: F > 10 (Stock & Yogo 2005). All tiers exceed F=150 comfortably.

**Note on DXY tier (F≈362 vs. extended F≈239):** The DXY tier excludes 2020–2022 (COVID, energy crisis), a period of extreme macro volatility that added noise to the first stage. Removing those 27 months sharpens the signal. This is expected and internally coherent — REER (F≈213, starts 1994) and commodities (F≈232, starts 1992) are both close to the full-sample F.

In [12]:
def test_f(df, controls, label):
    try:
        f = first_stage_f(df, controls=controls)
        s = 'Very strong' if f > 150 else ('Strong' if f > 50 else ('Marginal' if f > 10 else 'WEAK'))
        print(f'  {label:52s} | n={len(df):3d} | F={f:8.2f} | {s}')
        return f
    except Exception as e:
        print(f'  {label:52s} | ERROR: {e}'); return None

print('FIRST-STAGE F-STAT VALIDATION')
print('=' * 85)

ctrl_ext = BASE_CONTROLS + full_macro + VARIABLE_ROLES['controls_geopol']
# Note: BASE_CONTROLS still includes l2lwip for baseline replication, 
# but ctrl_ext uses VARIABLE_ROLES['controls_core'] which excludes it
test_f(df_baseline, BASE_CONTROLS,              '0. Baseline (core only, 4 vars)')
test_f(df_extended, VARIABLE_ROLES['controls_core'] + full_macro + VARIABLE_ROLES['controls_geopol'],
       '1. Extended (main spec, 14 vars)')
test_f(df_dxy,  VARIABLE_ROLES['controls_core'] + full_macro + VARIABLE_ROLES['controls_geopol'] + ['dxy'],
       '2. + DXY   (– 2019-11)')
test_f(df_reer, VARIABLE_ROLES['controls_core'] + full_macro + VARIABLE_ROLES['controls_geopol'] + ['reer'],
       '3. + REER  (1994-02 –)')
test_f(df_comm, VARIABLE_ROLES['controls_core'] + full_macro + VARIABLE_ROLES['controls_geopol'] + ['copper', 'wheat'],
       '4. + Commodities (1992-02 –)')

FIRST-STAGE F-STAT VALIDATION
  0. Baseline (core only, 4 vars)                      | n=385 | F=  243.40 | Very strong
  1. Extended (main spec, 14 vars)                     | n=385 | F=  242.70 | Very strong
  2. + DXY   (– 2019-11)                               | n=358 | F=  358.98 | Very strong
  3. + REER  (1994-02 –)                               | n=336 | F=  216.49 | Very strong
  4. + Commodities (1992-02 –)                         | n=360 | F=  236.30 | Very strong


236.29916070227767

## Cell 12: Falsification Check — UCT and GPR as Alternative Instruments

This is **not** a formal Hansen J overidentification test. UCT and GPR global are tested as a falsification exercise: if d2pri captures abrupt bilateral turning points, broader tension indices should predict PRI less precisely. Weak F is the expected and correct result.

In [13]:
print('FALSIFICATION CHECK — ALTERNATIVE INSTRUMENTS FOR PRI')
print('=' * 62)

check_cols = extended_cols + ['uct_tension_l1', 'gpr_global_l1']
df_overid  = df_merged[check_cols].dropna()
print(f'Check frame: n={len(df_overid)} | {df_overid.index.min().date()} to {df_overid.index.max().date()}')
print(f'(Reduced from 385 because UCT starts 1993-02 — expected)')
print()

alt_controls = VARIABLE_ROLES['controls_core'] + full_macro + VARIABLE_ROLES['controls_geopol']

for z, label in [('uct_tension_l1', 'UCT (Rogers et al. 2024)'),
                 ('gpr_global_l1',  'GPR global (Caldara-Iacoviello)')]:
    try:
        f = first_stage_f_alt(df_overid, z, alt_controls)
        s = 'Very strong' if f > 150 else ('Strong' if f > 50 else ('Marginal' if f > 10 else 'WEAK'))
        print(f'  {label:38s} | F={f:8.2f} | {s}')
    except Exception as e:
        print(f'  {label:38s} | ERROR: {e}')

print()
print('Interpretation: weak F for UCT/GPR is consistent with d2pri identifying')
print('a distinct, sharper channel than general geopolitical risk indices.')

FALSIFICATION CHECK — ALTERNATIVE INSTRUMENTS FOR PRI
Check frame: n=348 | 1993-03-31 to 2022-02-28
(Reduced from 385 because UCT starts 1993-02 — expected)

  UCT (Rogers et al. 2024)               | F=    1.70 | WEAK
  GPR global (Caldara-Iacoviello)        | F=    0.00 | WEAK

Interpretation: weak F for UCT/GPR is consistent with d2pri identifying
a distinct, sharper channel than general geopolitical risk indices.


## Cell 13: Instrument Orthogonality & Multicollinearity

**Orthogonality:** Verifies d2pri is not absorbed by geopolitical controls. |r| < 0.3 is the threshold.

**VIF:** Monitors collinearity in the extended control set. With `l2lwip` dropped, VIF should be substantially improved. `baa10y`/`gs10`/`vix`/`tb3ms` may still show elevated VIF (>10) due to financial variable co-movement, but this is expected and does not threaten identification given F>150.

In [14]:
print('INSTRUMENT ORTHOGONALITY')
print('=' * 58)
for alt in ['gpr_global_l1', 'gpr_chn_l1', 'uct_tension_l1']:
    if alt in df_overid.columns:
        sub = df_overid[['d2pri', alt]].dropna()
        r, p = stats.pearsonr(sub['d2pri'], sub[alt])
        flag = '  ← REVIEW' if abs(r) > 0.3 else '  ✓ orthogonal'
        print(f'  d2pri vs {alt:24s}: r={r:+.5f}  p={p:.4f}{flag}')

print()
print('VIF — EXTENDED CONTROLS (l2lwip DROPPED)')
print('=' * 52)
ctrl_cols = VARIABLE_ROLES['controls_core'] + full_macro + VARIABLE_ROLES['controls_geopol']
vif_df    = df_extended[ctrl_cols].replace([np.inf, -np.inf], np.nan).dropna()
print(f'Sample: {len(vif_df)} obs, {len(ctrl_cols)} controls')
print()
vif_data = pd.DataFrame({
    'variable': ctrl_cols,
    'VIF':      [variance_inflation_factor(vif_df.values, i) for i in range(len(ctrl_cols))]
}).sort_values('VIF', ascending=False)
vif_data['flag'] = vif_data['VIF'].apply(
    lambda v: 'HIGH' if v > 10 else ('moderate' if v > 5 else 'OK'))
print(vif_data.to_string(index=False))
print()
print('With l2lwip removed, VIF should be manageable.')
print('Remaining HIGH VIF (if any) from baa10y/gs10/vix/tb3ms: financial co-movement.')
print('Benign for IV given F>150, but flagged for DML regularization.')

INSTRUMENT ORTHOGONALITY
  d2pri vs gpr_global_l1           : r=-0.02054  p=0.7026  ✓ orthogonal
  d2pri vs gpr_chn_l1              : r=+0.00635  p=0.9060  ✓ orthogonal
  d2pri vs uct_tension_l1          : r=-0.02253  p=0.6754  ✓ orthogonal

VIF — EXTENDED CONTROLS (l2lwip DROPPED)
Sample: 385 obs, 14 controls

  variable       VIF     flag
     llwip 51.969479     HIGH
    baa10y 45.332277     HIGH
      gs10 32.328211     HIGH
       vix 19.940917     HIGH
     tb3ms 14.160788     HIGH
gpr_chn_l1  9.132057 moderate
gpr_usa_l1  5.348926 moderate
     brent  1.355396       OK
    indpro  1.330101       OK
       bdi  1.170532       OK
    dllgop  1.097629       OK
   dl2lgop  1.095489       OK
      gold  1.063914       OK
   cny_usd  1.022190       OK

With l2lwip removed, VIF should be manageable.
Remaining HIGH VIF (if any) from baa10y/gs10/vix/tb3ms: financial co-movement.
Benign for IV given F>150, but flagged for DML regularization.


## Cell 14: Coverage Report

In [15]:
def coverage_report(df, label):
    print(f'=== {label} (n={len(df)}) ===')
    rows = []
    for col in df.columns:
        if col in ['lwti', 'lpri', 'd2pri']: continue
        s  = df[col]
        nn = s.notna().sum()
        rows.append({
            'variable': col, 'n_obs': nn, 'n_total': len(s),
            'pct':  round(100 * nn / len(s), 1),
            'first': str(s.dropna().index.min().date()) if nn > 0 else '—',
            'last':  str(s.dropna().index.max().date()) if nn > 0 else '—'})
    rpt = pd.DataFrame(rows).sort_values('pct', ascending=False)
    print(rpt.to_string(index=False))
    return rpt

cov_ext  = coverage_report(df_extended, 'EXTENDED (main)')
cov_dxy  = coverage_report(df_dxy,      'DXY robustness')
cov_reer = coverage_report(df_reer,     'REER robustness')
cov_comm = coverage_report(df_comm,     'Commodities robustness')

=== EXTENDED (main) (n=385) ===
  variable  n_obs  n_total   pct      first       last
     llwip    385      385 100.0 1990-02-28 2022-02-28
    dllgop    385      385 100.0 1990-02-28 2022-02-28
   dl2lgop    385      385 100.0 1990-02-28 2022-02-28
       vix    385      385 100.0 1990-02-28 2022-02-28
      gs10    385      385 100.0 1990-02-28 2022-02-28
     tb3ms    385      385 100.0 1990-02-28 2022-02-28
    baa10y    385      385 100.0 1990-02-28 2022-02-28
     brent    385      385 100.0 1990-02-28 2022-02-28
      gold    385      385 100.0 1990-02-28 2022-02-28
       bdi    385      385 100.0 1990-02-28 2022-02-28
   cny_usd    385      385 100.0 1990-02-28 2022-02-28
    indpro    385      385 100.0 1990-02-28 2022-02-28
gpr_chn_l1    385      385 100.0 1990-02-28 2022-02-28
gpr_usa_l1    385      385 100.0 1990-02-28 2022-02-28
=== DXY robustness (n=358) ===
  variable  n_obs  n_total   pct      first       last
     llwip    358      358 100.0 1990-02-28 2019-11-30
  

## Cell 15: Save All Datasets

In [16]:
print('SAVING')
print('=' * 60)

for nm, df in {'df_baseline': df_baseline, 'df_extended': df_extended,
               'df_dxy': df_dxy, 'df_reer': df_reer, 'df_comm': df_comm}.items():
    p = FINAL / f'{nm}.csv'
    df.to_csv(p)
    print(f'  {nm:20s}: {df.shape[0]:3d} obs × {df.shape[1]:2d} vars → {p.name}')

with open(FINAL / 'variable_roles.json', 'w') as f:
    json.dump(VARIABLE_ROLES, f, indent=2)
print('  variable_roles.json')

for nm, rpt in [('extended', cov_ext), ('dxy', cov_dxy),
                ('reer', cov_reer),    ('comm', cov_comm)]:
    rpt.to_csv(FINAL / f'coverage_{nm}.csv', index=False)
print('  coverage reports (4 files)')
print('DONE.')

SAVING
  df_baseline         : 385 obs ×  6 vars → df_baseline.csv
  df_extended         : 385 obs × 17 vars → df_extended.csv
  df_dxy              : 358 obs × 18 vars → df_dxy.csv
  df_reer             : 336 obs × 18 vars → df_reer.csv
  df_comm             : 360 obs × 19 vars → df_comm.csv
  variable_roles.json
  coverage reports (4 files)
DONE.


## Summary

### Honest scope statement

This thesis does not claim "high-dimensional machine learning" with 50–100 predictors. That plan was a supervisor-drafted skeleton from before data exploration. After attempting downloads and inspecting coverage, the feasible specification is:

- **15 controls** (now 14 after dropping `l2lwip`) for the main extended dataset
- **Non-parametric control function estimation** via DML-PLIV, not "high-dimensional selection"
- **Single dyad** (US-China), not a 10–20 dyad panel
- **1990–2022 window**, not extended to 2026

This is a narrower but **defensible** contribution: relaxing linearity in the control function while maintaining Saadaoui's exact identification strategy. The ML component is the nuisance estimation (XGBoost for E[Y|X], E[D|X], E[Z|X]), not dimensionality expansion.

### Key design decisions

| Decision | Rationale |
|----------|-----------|
| Drop `l2lwip` | VIF=670k with `llwip`; numerical singularity in OLS, arbitrary selection in XGBoost |
| Drop `sentiment` | 54% missing; structurally unusable |
| Keep 1990–2022 | Preserves comparability with Saadaoui; avoids untestable structural breaks post-2022 |
| 14 controls, not 50–100 | n=385, n/p≈27.5; DML cross-fitting requires n >> p |
| Deterministic cache versioning | Replaces fragile heuristic; reproducible across runs |
| Lag before dropna in gap tiers | Fixes interior-NaN corruption from previous version |

### Key outputs

| Metric | Value |
|--------|-------|
| Main sample (`df_extended`) | 385 obs, 1990-02 to 2022-02 |
| Controls | 3 core + 9 macro + 2 geopol = 14 |
| Baseline F | ~236 |
| Extended F | ~239 (expected; l2lwip drop should not affect F much) |
| d2pri – gpr_chn_l1 correlation | r ≈ +0.006, p = 0.91 |

### Next notebooks
- **Notebook 06:** LP-IV estimation (baseline + extended), h = 0…48
- **Notebook 07:** Robustness — tier subsamples, alternative controls
- **Notebook 08:** State-dependent LP — VIX and UCT interaction terms
- **Notebook 09:** Double ML — XGBoost nuisance estimation, SHAP importance, comparison with linear IV

In [17]:
import doubleml; print(f'doubleml: {doubleml.__version__}')
import xgboost; print(f'xgboost: {xgboost.__version__}')
import shap; print(f'shap: {shap.__version__}')

doubleml: 0.11.2
xgboost: 3.0.2
shap: 0.47.2


In [18]:
# ── Cell 16 : NLP tier A  –  GDELT structured features (full 1990-2022 coverage) ──

import pandas as pd
import numpy as np
from pathlib import Path

ROOT  = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FINAL = ROOT / 'data' / 'final'
NLP   = ROOT / 'data' / '03_nlp'

# ── 1. Load base dataset ──────────────────────────────────────────────────────
df_ext = pd.read_csv(FINAL / 'df_extended.csv', index_col=0, parse_dates=True)
df_ext.index = pd.to_datetime(df_ext.index).to_period('M').to_timestamp('M')
print(f"df_extended loaded : {df_ext.shape}  |  index type: {type(df_ext.index[0])}")

# ── 2. Load GDELT NLP feature matrix ─────────────────────────────────────────
nlp_path = NLP / 'feature_matrix_nlp_A.csv'
df_nlp = pd.read_csv(nlp_path, index_col=0, parse_dates=True)

# THE BUG FIX: normalise both indices to month-end timestamps (same as df_ext)
df_nlp.index = pd.to_datetime(df_nlp.index).to_period('M').to_timestamp('M')
print(f"GDELT NLP loaded   : {df_nlp.shape}  |  index type: {type(df_nlp.index[0])}")

# ── 3. Select GDELT columns with ≥ 90 % coverage ─────────────────────────────
gdelt_candidates = [c for c in df_nlp.columns if c.startswith('gdelt_')]
coverage = df_nlp[gdelt_candidates].notna().mean()
gdelt_primary = coverage[coverage >= 0.90].index.tolist()
print(f"\nGDELT columns with ≥90% coverage: {len(gdelt_primary)}")
for col in gdelt_primary:
    print(f"  {col:45s} coverage={coverage[col]:.1%}  "
          f"range=[{df_nlp[col].min():.3f}, {df_nlp[col].max():.3f}]")

# ── 4. Also pull GPR and WUI if present ──────────────────────────────────────
extra_nlp = [c for c in ['gpr', 'ea_gpr', 'wui'] if c in df_nlp.columns
             and df_nlp[c].notna().mean() >= 0.70]
print(f"\nExtra NLP indices included: {extra_nlp}")

nlp_cols = gdelt_primary + extra_nlp

# ── 5. Merge ──────────────────────────────────────────────────────────────────
df_extended_nlp = df_ext.join(df_nlp[nlp_cols], how='left')

# Verify merge worked
n_null_after = df_extended_nlp[nlp_cols].isna().sum()
print(f"\nPost-merge null counts (should be near 0 for GDELT):")
print(n_null_after[n_null_after > 0].to_string() if n_null_after.any() else "  All zeros — merge successful.")

print(f"\ndf_extended_nlp : {df_extended_nlp.shape}")
print(f"  Base vars      : {df_ext.shape[1]}")
print(f"  NLP vars added : {len(nlp_cols)}")

df_extended loaded : (385, 17)  |  index type: <class 'pandas._libs.tslibs.timestamps.Timestamp'>
GDELT NLP loaded   : (386, 81)  |  index type: <class 'pandas._libs.tslibs.timestamps.Timestamp'>

GDELT columns with ≥90% coverage: 10
  gdelt_total_events_log                        coverage=99.7%  range=[3.135, 10.047]
  gdelt_goldstein_mean                          coverage=99.7%  range=[-0.733, 4.789]
  gdelt_sentiment_signal                        coverage=99.7%  range=[0.107, 27.899]
  gdelt_conflict_share                          coverage=99.7%  range=[0.000, 0.091]
  gdelt_hostility_share                         coverage=99.7%  range=[0.000, 0.286]
  gdelt_topic_pca_1                             coverage=99.7%  range=[-5.873, 4.449]
  gdelt_topic_pca_2                             coverage=99.7%  range=[-3.209, 7.756]
  gdelt_topic_pca_3                             coverage=99.7%  range=[-7.837, 5.247]
  gdelt_topic_pca_4                             coverage=99.7%  range=[-3.936, 6

In [19]:
# ── Cell 17 : NLP tier B  –  FinBERT + BERTopic (2015-2022 short sample) ────

bert_path = ROOT / 'data' / 'feature_matrix_nlp_AB.csv'

if bert_path.exists():
    df_ab = pd.read_csv(bert_path, index_col=0, parse_dates=True)
    df_ab.index = pd.to_datetime(df_ab.index).to_period('M').to_timestamp('M')

    bert_cols = [c for c in ['finbert_net', 'finbert_neg_mean', 'finbert_pos_mean',
                              'bertopic_conflict', 'cny_vol']
                 if c in df_ab.columns]
    coverage_b = df_ab[bert_cols].notna().mean()
    print("FinBERT / BERTopic coverage:")
    for col in bert_cols:
        print(f"  {col:30s}  {coverage_b[col]:.1%}  ({int(coverage_b[col]*len(df_ab))}/{len(df_ab)} months)")

    # Merge into the NLP-A frame to get the combined tier
    df_extended_nlp_bert = df_extended_nlp.join(df_ab[bert_cols], how='left')
    short_sample = df_extended_nlp_bert[bert_cols].notna().all(axis=1)
    print(f"\ndf_extended_nlp_bert : {df_extended_nlp_bert.shape}")
    print(f"Short-sample window (all FinBERT cols non-null): {short_sample.sum()} months "
          f"({df_extended_nlp_bert[short_sample].index.min().strftime('%Y-%m')} – "
          f"{df_extended_nlp_bert[short_sample].index.max().strftime('%Y-%m')})")
else:
    print("feature_matrix_nlp_AB.csv not found – FinBERT tier skipped.")
    df_extended_nlp_bert = df_extended_nlp.copy()
    bert_cols = []
    short_sample = pd.Series(False, index=df_extended_nlp_bert.index)


FinBERT / BERTopic coverage:
  finbert_net                     21.2%  (82/386 months)
  finbert_neg_mean                21.2%  (82/386 months)
  finbert_pos_mean                21.2%  (82/386 months)
  bertopic_conflict               21.5%  (83/386 months)

df_extended_nlp_bert : (385, 34)
Short-sample window (all FinBERT cols non-null): 82 months (2015-04 – 2022-02)


In [20]:
# ── Cell 18 : Save NLP tiers + variable_roles update ─────────────────────────
import json

# Save files
df_extended_nlp.to_csv(FINAL / 'df_extended_nlp.csv')
df_extended_nlp_bert.to_csv(FINAL / 'df_extended_nlp_bert.csv')
print("Saved:")
print(f"  df_extended_nlp      : {df_extended_nlp.shape}  → {FINAL / 'df_extended_nlp.csv'}")
print(f"  df_extended_nlp_bert : {df_extended_nlp_bert.shape}  → {FINAL / 'df_extended_nlp_bert.csv'}")

# Update variable_roles.json
with open(FINAL / 'variable_roles.json') as f:
    roles = json.load(f)

roles['controls_nlp_gdelt']    = gdelt_primary
roles['controls_nlp_extra']    = extra_nlp
roles['controls_nlp_bert']     = bert_cols
roles['nlp_short_sample_start'] = str(df_extended_nlp_bert[short_sample].index.min())[:7] if short_sample.any() else None
roles['nlp_short_sample_end']   = str(df_extended_nlp_bert[short_sample].index.max())[:7] if short_sample.any() else None

with open(FINAL / 'variable_roles.json', 'w') as f:
    json.dump(roles, f, indent=2)
print("  variable_roles.json updated")

# Coverage audit
print("\n── FINAL COVERAGE AUDIT ──────────────────────────────────────────────────")
for tier_name, tier_df, tier_cols in [
    ('df_extended (baseline)',  df_ext,              list(df_ext.columns)),
    ('df_extended_nlp (GDELT)', df_extended_nlp,     nlp_cols),
    ('df_extended_nlp_bert',    df_extended_nlp_bert, bert_cols),
]:
    if not tier_cols:
        continue
    cov = tier_df[tier_cols].notna().mean()
    print(f"\n{tier_name}  ({tier_df.shape[0]} obs)")
    print(f"  Min coverage : {cov.min():.1%}  ({cov.idxmin()})")
    print(f"  Med coverage : {cov.median():.1%}")
    print(f"  Vars ≥ 90%   : {(cov>=0.90).sum()} / {len(tier_cols)}")



Saved:
  df_extended_nlp      : (385, 30)  → c:\Users\HP\Desktop\replication+contribution\data\final\df_extended_nlp.csv
  df_extended_nlp_bert : (385, 34)  → c:\Users\HP\Desktop\replication+contribution\data\final\df_extended_nlp_bert.csv
  variable_roles.json updated

── FINAL COVERAGE AUDIT ──────────────────────────────────────────────────

df_extended (baseline)  (385 obs)
  Min coverage : 100.0%  (lwti)
  Med coverage : 100.0%
  Vars ≥ 90%   : 17 / 17

df_extended_nlp (GDELT)  (385 obs)
  Min coverage : 75.1%  (ea_gpr)
  Med coverage : 100.0%
  Vars ≥ 90%   : 12 / 13

df_extended_nlp_bert  (385 obs)
  Min coverage : 21.3%  (finbert_net)
  Med coverage : 21.3%
  Vars ≥ 90%   : 0 / 4
